---------------------------------------------------------------- shared setup (same in every engine notebook)
 THE TOOLKIT -- what this cell leaves behind. The same nine functions and four names are
 in every engine notebook of the ladder, so what you learn here works in v2 through v8.
 (In a cell, run  help(play_match)  to see any of these again without scrolling back.)

 NAMES YOU CAN READ
   IN_COLAB        True on Google Colab. When True this cell also apt-installs Stockfish.
   STOCKFISH_PATH  Empty by default -- paste your own path here if the finder below fails.
   STOCKFISH       Full path to the Stockfish binary, or None. Printed on the last line.
   VALUES          Piece values in centipawns: pawn 100, knight 300, bishop 300,
                   rook 500, queen 900, king 0 (the king is never captured, so it scores 0).

 OPEN AN ENGINE, SEE A BOARD
   stockfish(elo=None, threads=1)
                   Start a Stockfish process and return it. Pass elo=1350 to weaken it to
                   a club player. ALWAYS call .quit() on what it returns, or the process
                   stays alive after the cell ends.
   show(board, size=320)
                   Draw the board in the notebook with the last move highlighted.
                   Displays; returns nothing.

 PLAY GAMES BETWEEN TWO BOTS
   A "bot" here is any function that takes a board and returns a legal move --
   greedy_bot, random_bot, and the turing_bot you write in Section 5 all qualify.
   play_game(white_fn, black_fn, max_plies=160, start=None)
                   Play one game; returns the finished board. start takes a FEN if you
                   want to begin somewhere other than move 1.
   play_match(fn_a, fn_b, games=10, max_plies=160, seed=1)
                   Play a match with colours alternating; returns (points_for_a, decisive)
                   where a win is 1 point, a draw 0.5, and decisive counts games that
                   actually finished in a result. This is how you measure a change.

 BOTS AND SCORING -- your opponents and your yardstick
   random_bot(board)
                   Plays a legal move at random. The floor: anything you write should beat it.
   material(board, color)
                   Centipawn material balance from color's point of view (+ is good for color).
   greedy_bot(board)
                   The Setup-Week baseline, three rules: play mate if you see it, else take
                   the most material, else choose at random among equals. Scores about
                   26-28 out of 30 against random_bot (it moves with the seed) --
                   beat THIS and you have built something.

 SAVE YOUR EVIDENCE -- both write into the portfolio layout when it exists, else here
   save_pgn(board, name, white="?", black="?")
                   Write a finished game to games/<name>; returns the path it used.
   save_results(rows, header, engine_folder)
                   Write engines/<engine_folder>/results.csv; returns the path it used.
                   rows is a list of lists, header a list of column names.
----------------------------------------------------------------------------------------

In [ ]:
# At this point, this should be installed, if not, uncomment to run and install

#%pip install chess -q

In [1]:
# --- Imports --- #

import sys, os, shutil, csv, time, random, platform, re
import chess, chess.engine, chess.pgn, chess.svg
from IPython.display import display, SVG, clear_output
import asyncio
import sys
import numpy as np

# --- System Setup --- #

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# --- Stockfish Setup --- #

STOCKFISH_PATH = "C:\\Users\\Kenny\\Documents\\University Of Toledo\\Fall 26\\MATH 4900\\stockfish\\stockfish-windows-x86-64-universal.exe"      # laptop users: paste the full path here if the finder below cannot see it
STOCKFISH = next((p for p in [STOCKFISH_PATH, os.environ.get("STOCKFISH_PATH"), shutil.which("stockfish"),
                              "/usr/games/stockfish", "/usr/local/bin/stockfish", "/opt/homebrew/bin/stockfish",
                              "./stockfish", "./stockfish.exe"] if p and os.path.isfile(p)), None)

def stockfish(elo=None, threads=1):
    """Open a Stockfish process (optionally rating-limited). Remember to .quit() it."""
    if not STOCKFISH:
        raise FileNotFoundError("Stockfish not found -- set STOCKFISH_PATH (see Setup Week Kit, Step 5.1)")
    eng = chess.engine.SimpleEngine.popen_uci(STOCKFISH)
    if elo is not None:
        opt = eng.options["UCI_Elo"]
        eng.configure({"UCI_LimitStrength": True, "UCI_Elo": max(opt.min, min(opt.max, elo))})
    return eng


sf_eng = stockfish(elo=1350)


print("python-chess", chess.__version__, "| Stockfish:", STOCKFISH or "NOT FOUND (only the engine cells need it)")









C:\Users\Kenny\AppData\Local\Temp\ipykernel_52452\1615443695.py:13: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\Kenny\AppData\Local\Temp\ipykernel_52452\1615443695.py:13: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


python-chess 1.11.2 | Stockfish: C:\Users\Kenny\Documents\University Of Toledo\Fall 26\MATH 4900\stockfish\stockfish-windows-x86-64-universal.exe


In [2]:
# --- Helper Functions --- #
def count_files(folder_path, extension):
    ''' 
    Count how many files of a given extension exist in a directory.
        Args: 
            folder_path: string
                Path string to the target directory
            extension: string
                Extension string to match (.csv, .pgn)
        Returns:
            Integer count of matching files
    '''
    if not os.path.exists(folder_path):
        return 0

    ext = extension.lower()
    if not ext.startswith('.'):
        ext = f'.{ext}'

    return len(
        [ # This is just list comprehension
            f
            for f in os.listdir(folder_path)
            if f.lower().endswith(ext)
            and os.path.isfile(os.path.join(folder_path, f))
        ]
    )
# end count_files()

def get_next_match_number(folder_path):
    """Scan existing .pgn files in folder_path, find the max Match number, and return max + 1."""
    if not os.path.exists(folder_path):
        return 1

    pattern = re.compile(r"^Match(\d+)", re.IGNORECASE)
    match_nums = [
        int(pattern.search(f).group(1))
        for f in os.listdir(folder_path)
        if f.lower().endswith(".pgn") and pattern.search(f)
    ]

    return max(match_nums) + 1 if match_nums else 1
# end get_next_match_number()

def save_results(
    rows,
    header=["experiment", "games", "points_for_change", "decisive"],
    white="?",
    black="?",
    ):
    
    target_dir = os.path.join("..", "results")
    os.makedirs(target_dir, exist_ok=True)

    # Name the file by the matchup sorted alphabetically
    p1, p2 = sorted([str(white), str(black)])
    path = os.path.join(target_dir, f"{p1}_vs_{p2}.csv")

    # Check if file exists to determine if the header is needed
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(header)
        w.writerows(rows)

    print(f"Results appended to {path}")
    return path
# end save_results()

def save_pgn(board, name, target_dir, white="?", black="?"):
    """Save an individual game as PGN into ../games/<subfolder>/."""
    
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "Gambit Lab"
    game.headers["White"] = str(white)
    game.headers["Black"] = str(black)
    game.headers["Result"] = board.result() if board.is_game_over() else "*"

    file_path = os.path.join(target_dir, name)
    with open(file_path, "w", encoding="utf-8") as f:
        print(game, file=f, end="\n\n")

    print(f"saved game -> {file_path}")
    return file_path
# end save_pgn()

def material(board, color):
    """Material balance in centipawns from `color`'s point of view: positive means `color` is ahead."""
    return sum(VALUES[p.piece_type] * (1 if p.color == color else -1) for p in board.piece_map().values())
# end material()

In [9]:
# --- Chess Game Engine --- #

VALUES = {chess.PAWN: 100, chess.KNIGHT: 300, chess.BISHOP: 300, chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}

def show(board, size=320):
    """Draw `board` in the notebook, highlighting the last move. Displays; returns nothing."""
    display(SVG(chess.svg.board(board, size=size, lastmove=board.peek() if board.move_stack else None)))

def play_game(white_fn, black_fn, max_plies=160, start=None):
    """Two Python functions (board -> move) play a game. Returns the final board."""
    b = chess.Board(start) if start else chess.Board()
    while not b.is_game_over() and len(b.move_stack) < max_plies:
        b.push((white_fn if b.turn == chess.WHITE else black_fn)(b))
    return b

def play_match(fn_a, fn_b, games=10, max_plies=160, seed=1, save_games=True, save_result=True):
    """fn_a vs fn_b with colours alternating. Returns (points for A, decisive games)."""
    pts, decisive = 0.0, 0

    name_a = getattr(fn_a, "__name__", "Bot_A")
    name_b = getattr(fn_b, "__name__", "Bot_B")

    # Determine subfolder structure
    p1, p2 = sorted([str(name_a), str(name_b)])
    subfolder = f"{p1}_vs_{p2}"
    target_dir = os.path.join("..", "games", subfolder)

    if save_games:
        os.makedirs(target_dir, exist_ok=True)
        match_num = get_next_match_number(target_dir)

    # Determine padding width based on total game count (e.g., 01-10 vs 001-100)
    pad_width = len(str(games))

    for i in range(games):
        random.seed(seed + i)
        a_white = i % 2 == 0

        white_player = name_a if a_white else name_b
        black_player = name_b if a_white else name_a

        b = (
            play_game(fn_a, fn_b, max_plies)
            if a_white
            else play_game(fn_b, fn_a, max_plies)
        )
        res = b.result() if b.is_game_over() else "1/2-1/2"

        if res != "1/2-1/2" and b.is_game_over():
            decisive += 1
        if res == "1-0":
            pts += 1 if a_white else 0
        elif res == "0-1":
            pts += 0 if a_white else 1
        else:
            pts += 0.5

        if save_games:
            # Construct a per-game indexed filename: e.g., "round1_game_01"
            game_num = str(i + 1).zfill(pad_width)
            game_filename = f"Match{match_num:02d}_Game{game_num}.pgn"

            save_pgn(
                board=b,
                name=game_filename,
                white=white_player,
                black=black_player,
                target_dir=target_dir,
            )

    if save_result:
        save_results(
            rows = [[subfolder, 20, pts, decisive]],
            white = white_player,
            black = black_player
            )

    return pts, decisive

# --- Bot Making Function --- #

def make_eval_bot(eval_fn, name=None):
    """Factory that creates a 1-ply bot function using any eval_fn(board, me)."""

    def bot(board):
        me = board.turn
        best = None
        moves = []

        for m in board.legal_moves:
            board.push(m)
            s = 10**6 if board.is_checkmate() else eval_fn(board, me)
            board.pop()

            if best is None or s > best:
                best, moves = s, [m]
            elif s == best:
                moves.append(m)

        return random.choice(moves)

    # Set __name__ so play_match and folder/file saving can read it automatically
    bot.__name__ = name if name else eval_fn.__name__.replace("eval", "bot")
    return bot
# end make_eval_bot

In [13]:
# --- Eval Functions --- #

def turing_eval(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score = 100 * (mine + 100) / (theirs + 100)          # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 20
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score
# end turing_eval()

def turing_eval_mobile(board, me):
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)

    score = 100 * (mine + 100) / (theirs + 100)

    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 1.0 * (my_moves - their_moves)               # <-- THE ONE CHANGE (was 0.5)
    if board.turn == me and board.is_check(): score -= 20
    score += 5 if board.has_castling_rights(me) else 0
    return score
# end turing_eval_mobile()

def turing_eval_sqrt_prod(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score = np.sqrt(mine) * np.sqrt(theirs) #100 * (mine + 100) / (theirs + 100)          # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 20
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score
# end turing_eval_sqrt_prod()

def turing_eval_must_lose(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score = -( 100 * (mine + 100) / (theirs + 100) )         # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 200
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score
# end turing_eval_must_lose

def turing_eval_diff(board, me):
    """A Turing-style evaluation from `me`'s point of view. Bigger is better for `me`."""
    mine  = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color == me and p.piece_type != chess.KING)
    theirs = sum(VALUES[p.piece_type] for p in board.piece_map().values() if p.color != me and p.piece_type != chess.KING)
    score =  mine - theirs          # Turing's material RATIO (+100 keeps it finite)
    # mobility: how many moves would I have here?
    saved = board.turn
    board.turn = me;      my_moves = board.legal_moves.count()
    board.turn = not me;  their_moves = board.legal_moves.count()
    board.turn = saved
    score += 0.5 * (my_moves - their_moves)
    # king safety: penalty if my king is in check right now
    if board.turn == me and board.is_check():
        score -= 20
    # castling rights kept (a proxy for 'can still castle')
    score += 5 if board.has_castling_rights(me) else 0
    return score
# end turing_eval()

In [14]:
# --- Bots --- #
def greedy_bot(board):
    """Mate if you see it, else grab the most material, random tie-break."""
    me, best, moves = board.turn, None, []
    for m in board.legal_moves:
        board.push(m); s = 10**6 if board.is_checkmate() else material(board, me); board.pop()
        if best is None or s > best: best, moves = s, [m]
        elif s == best: moves.append(m)
    return random.choice(moves)
# end greedy_bot()

def random_bot(board):
    """A legal move chosen at random. The weakest possible opponent -- your baseline's baseline."""
    return random.choice(list(board.legal_moves))
# end random_bot()

def sf_bot(board): # Stockfish
    return sf_eng.play(board, chess.engine.Limit(time=0.1)).move
# end sf_bot()

# --- Bots made with make_eval_bot() --- #
turing_bot = make_eval_bot(turing_eval,"Turing_Bot")
turing_bot_mobile = make_eval_bot(turing_eval_mobile,'Turing_Bot_Mobile')
sqrt_bot = make_eval_bot(turing_eval_sqrt_prod,'Sqrt_Bot')
must_lose = make_eval_bot(turing_eval_must_lose,'Must_Lose')
diff = make_eval_bot(turing_eval_diff,'Turing_Bot_Difference')

# Play Games Below Here #

In [10]:
play_match(must_lose,turing_bot)

saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game01.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game02.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game03.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game04.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game05.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game06.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game07.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game08.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game09.pgn
saved game -> ..\games\Must_Lose_vs_Turing_Bot\Match03_Game10.pgn
Results appended to ..\results\Must_Lose_vs_Turing_Bot.csv


(2.0, 6)

In [11]:
play_match(must_lose,sf_bot)

saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game01.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game02.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game03.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game04.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game05.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game06.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game07.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game08.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game09.pgn
saved game -> ..\games\Must_Lose_vs_sf_bot\Match01_Game10.pgn
Results appended to ..\results\Must_Lose_vs_sf_bot.csv


(0.0, 10)

In [12]:
play_match(sqrt_bot,sf_bot)

saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game01.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game02.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game03.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game04.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game05.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game06.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game07.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game08.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game09.pgn
saved game -> ..\games\Sqrt_Bot_vs_sf_bot\Match01_Game10.pgn
Results appended to ..\results\Sqrt_Bot_vs_sf_bot.csv


(0.0, 10)

In [15]:
play_match(diff,turing_bot)

saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game01.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game02.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game03.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game04.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game05.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game06.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game07.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game08.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game09.pgn
saved game -> ..\games\Turing_Bot_vs_Turing_Bot_Difference\Match01_Game10.pgn
Results appended to ..\results\Turing_Bot_vs_Turing_Bot_Difference.csv


(6.0, 6)